In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import json
import time
import numpy as np
import pandas as pd
import scipy.io as sio
import pandas as pd, numpy as np, os, glob
import wandb
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)


In [3]:
csv_path = '/content/drive/MyDrive/Ketastasia/data/combined.csv'
df = pd.read_csv(csv_path)

In [4]:
train_df = df[df['split'] == 'train'].reset_index(drop=True)

print(f"სულ ბაზაშია: {len(df)} ვიდეო")
print(f"სატრენინგო (train) ნაწილშია: {len(train_df)} ვიდეო\n")
print("=== დიაგნოსტიკის შედეგები ===")

# 4. თქვენი ორიგინალი კოდი imbalance-ის დასათვლელად:
counts = train_df['label'].value_counts()
print(counts)

print(f"\nყველაზე დიდი კლასი: {counts.max()}, ყველაზე პატარა: {counts.min()}")
print(f"Imbalance ratio (max/min): {counts.max()/counts.min():.1f}x")

# კლასები რომლებსაც ძალიან ცოტა ვიდეო აქვთ
rare_classes = counts[counts < 15].index.tolist()
print(f"\n<15 ვიდეო (სახიფათო კლასები): {rare_classes}")

სულ ბაზაშია: 1813 ვიდეო
სატრენინგო (train) ნაწილშია: 1295 ვიდეო

=== დიაგნოსტიკის შედეგები ===
label
pushup                 191
squat                  186
pullup                 160
bench_press            143
jumping_jacks           82
situp                   71
clean_and_jerk          64
jump_rope               56
bicep_curl              44
lat_pulldown            36
tricep_pushdown         35
lateral_raise           26
deadlift                24
incline_bench_press     24
chest_fly               21
leg_extension           19
t_bar_row               16
leg_raises              15
tricep_dips             14
hammer_curl             13
hip_thrust              13
shoulder_press          12
russian_twist           10
romanian_deadlift        8
decline_bench_press      7
plank                    5
Name: count, dtype: int64

ყველაზე დიდი კლასი: 191, ყველაზე პატარა: 5
Imbalance ratio (max/min): 38.2x

<15 ვიდეო (სახიფათო კლასები): ['tricep_dips', 'hammer_curl', 'hip_thrust', 'shoulder_press', 

In [5]:
wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
# CELL 2 — თუ Penn/Kaggle მონაცემები ჯერ არ არის extract-ილი ამ session-ში
!tar -xzf '/content/drive/MyDrive/Ketastasia/data/Penn_Action.tar.gz' -C '/content/'
!unzip -o -q '/content/drive/MyDrive/Ketastasia/data/kaggle.zip' -d '/content/Kaggle'

In [11]:
# CELL 3 — Penn Action-ის ფილტრი fitness-actions-ზე + path
FITNESS_ACTIONS = ['squat','pushup','pullup','jumping_jacks','situp',
                    'bench_press','jump_rope','clean_and_jerk']

LABELS_DIR = '/content/Penn_Action/labels'
penn_rows = []

for mat_file in sorted(glob.glob(f'{LABELS_DIR}/*.mat')):
    ann = sio.loadmat(mat_file)
    action = ann['action'][0].strip()
    if action not in FITNESS_ACTIONS:
        continue
    video_id = os.path.basename(mat_file).replace('.mat', '')
    frames_dir = f'/content/Penn_Action/frames/{video_id}'
    penn_rows.append({
        'video_id': video_id, 'source': 'penn', 'label': action,
        'path': frames_dir, 'media_type': 'frames_dir'
    })

penn_meta = pd.DataFrame(penn_rows)
print(f"Penn: {len(penn_meta)} ვიდეო")
print(penn_meta['label'].value_counts())

Penn: 1163 ვიდეო
label
squat             231
pushup            211
pullup            199
bench_press       140
jumping_jacks     112
situp             100
clean_and_jerk     88
jump_rope          82
Name: count, dtype: int64


In [12]:
# CELL 4 — Kaggle-ის label-mapping + path
LABEL_MAP = {
    'squat': 'squat', 'push-up': 'pushup', 'pull Up': 'pullup',
    'bench press': 'bench_press', 'leg raises': 'leg_raises',
    'barbell biceps curl': 'bicep_curl', 'chest fly machine': 'chest_fly',
    'leg extension': 'leg_extension', 'lateral raise': 'lateral_raise',
    'hammer curl': 'hammer_curl', 'decline bench press': 'decline_bench_press',
    'lat pulldown': 'lat_pulldown', 'incline bench press': 'incline_bench_press',
    'tricep Pushdown': 'tricep_pushdown', 'romanian deadlift': 'romanian_deadlift',
    'deadlift': 'deadlift', 't bar row': 't_bar_row', 'russian twist': 'russian_twist',
    'hip thrust': 'hip_thrust', 'plank': 'plank', 'shoulder press': 'shoulder_press',
    'tricep dips': 'tricep_dips'
}

KAGGLE_DIR = '/content/Kaggle'
kaggle_rows = []

for action_folder in os.listdir(KAGGLE_DIR):
    action_path = os.path.join(KAGGLE_DIR, action_folder)
    if not os.path.isdir(action_path):
        continue
    label = LABEL_MAP.get(action_folder, action_folder.lower().replace(' ', '_'))
    video_files = glob.glob(f'{action_path}/*.mp4') + glob.glob(f'{action_path}/*.MOV')
    for vf in video_files:
        video_id = os.path.basename(vf).rsplit('.', 1)[0]
        kaggle_rows.append({
            'video_id': video_id, 'source': 'kaggle', 'label': label,
            'path': vf, 'media_type': 'video_file'
        })

kaggle_meta = pd.DataFrame(kaggle_rows)
print(f"Kaggle: {len(kaggle_meta)} ვიდეო")
print(kaggle_meta['label'].value_counts())

Kaggle: 650 ვიდეო
label
bicep_curl             62
bench_press            61
pushup                 56
lat_pulldown           51
tricep_pushdown        50
lateral_raise          37
incline_bench_press    33
deadlift               32
squat                  29
chest_fly              28
pullup                 26
leg_extension          25
t_bar_row              21
leg_raises             21
tricep_dips            20
hammer_curl            19
hip_thrust             18
shoulder_press         17
russian_twist          13
romanian_deadlift      12
decline_bench_press    12
plank                   7
Name: count, dtype: int64


In [13]:
# CELL 5 — გაერთიანება + შენახვა (ეს არის Notebook 0-ის input Section 2-ისთვის)
video_meta = pd.concat([penn_meta, kaggle_meta], ignore_index=True)
print(f"სულ: {len(video_meta)} ვიდეო, {video_meta['label'].nunique()} კლასი")

video_meta.to_csv('/content/drive/MyDrive/Ketastasia/data/combined_with_paths.csv', index=False)
video_meta.head()

სულ: 1813 ვიდეო, 26 კლასი


,video_id,source,label,path,media_type
0,0341,penn,bench_press,/content/Penn_Action/frames/0341,frames_dir
1,0342,penn,bench_press,/content/Penn_Action/frames/0342,frames_dir
2,0343,penn,bench_press,/content/Penn_Action/frames/0343,frames_dir
3,0344,penn,bench_press,/content/Penn_Action/frames/0344,frames_dir
4,0345,penn,bench_press,/content/Penn_Action/frames/0345,frames_dir


In [18]:
# CELL 6 (განახლებული) — label remapping
LABEL_MERGE_MAP = {
    'incline_bench_press': 'bench_press',
    'decline_bench_press': 'bench_press',
    'romanian_deadlift':   'deadlift',
}
EXCLUDE_LABELS = ['plank']

video_meta = pd.read_csv('/content/drive/MyDrive/Ketastasia/data/combined_with_paths.csv')

video_meta = video_meta[~video_meta['label'].isin(EXCLUDE_LABELS)].copy()
video_meta['label'] = video_meta['label'].replace(LABEL_MERGE_MAP)

print(f"კლასების რაოდენობა: {video_meta['label'].nunique()}")
print(video_meta['label'].value_counts())

კლასების რაოდენობა: 22
label
pushup             267
squat              260
bench_press        246
pullup             225
jumping_jacks      112
situp              100
clean_and_jerk      88
jump_rope           82
bicep_curl          62
lat_pulldown        51
tricep_pushdown     50
deadlift            44
lateral_raise       37
chest_fly           28
leg_extension       25
leg_raises          21
t_bar_row           21
tricep_dips         20
hammer_curl         19
hip_thrust          18
shoulder_press      17
russian_twist       13
Name: count, dtype: int64


In [19]:
# CELL 7 --split
from sklearn.model_selection import StratifiedGroupKFold

def make_video_split(meta_df, random_state=42):
    # 5 სფლიტი ტესტისთვის (~20%)
    sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_state)
    train_val_idx, test_idx = next(sgkf_test.split(
        meta_df, meta_df['label'], groups=meta_df['video_id']))

    train_val_meta = meta_df.iloc[train_val_idx].reset_index(drop=True)
    test_meta = meta_df.iloc[test_idx].reset_index(drop=True)

    # 4 სფლიტი ვალიდაციისთვის დარჩენილიდან
    sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_state)
    train_idx, val_idx = next(sgkf_val.split(
        train_val_meta, train_val_meta['label'], groups=train_val_meta['video_id']))

    train_meta = train_val_meta.iloc[train_idx]
    val_meta = train_val_meta.iloc[val_idx]

    split_map = {}
    split_map.update({vid: 'train' for vid in train_meta['video_id']})
    split_map.update({vid: 'val'   for vid in val_meta['video_id']})
    split_map.update({vid: 'test'  for vid in test_meta['video_id']})
    return split_map

split_map = make_video_split(video_meta)
video_meta['split'] = video_meta['video_id'].map(split_map)

# შემოწმება
check = video_meta.groupby(['split', 'label']).size().unstack(fill_value=0)
print(check.T)

split            test  train  val
label                            
bench_press        49    148   49
bicep_curl         13     36   13
chest_fly           5     18    5
clean_and_jerk     18     53   17
deadlift            7     26   11
hammer_curl         5     11    3
hip_thrust          4     11    3
jump_rope          14     51   17
jumping_jacks      23     66   23
lat_pulldown       12     30    9
lateral_raise       7     22    8
leg_extension       5     15    5
leg_raises          4     13    4
pullup             46    133   46
pushup             54    161   52
russian_twist       2      8    3
shoulder_press      3     10    4
situp              20     61   19
squat              52    155   53
t_bar_row           5     13    3
tricep_dips         5     10    5
tricep_pushdown     9     32    9


In [20]:
# CELL 8 — საბოლოო manifest
video_meta.to_csv('/content/drive/MyDrive/Ketastasia/data/combined_split_final.csv', index=False)
print(video_meta['split'].value_counts())

split
train    1083
test      362
val       361
Name: count, dtype: int64


In [21]:
# CELL 9 — decord install
!pip install decord -q
import decord
from decord import VideoReader, cpu
decord.bridge.set_bridge('native')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 31.3 MB/s eta 0:00:00


In [22]:
# CELL 10 — cache config
CACHE_DIR = '/content/frame_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

NUM_CACHE_FRAMES = 32
CACHE_SIZE = 256

In [23]:
# CELL 11 — უნიფიცირებული, სწრაფი frame loader
import cv2

def load_uniform_frames(row, num_frames=NUM_CACHE_FRAMES, size=CACHE_SIZE):
    if row['media_type'] == 'frames_dir':
        frame_files = sorted(glob.glob(os.path.join(row['path'], '*.jpg')))
        total = len(frame_files)
        if total == 0:
            return None
        idxs = np.linspace(0, total - 1, num_frames).astype(int)
        frames = []
        for i in idxs:
            img = cv2.imread(frame_files[i])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (size, size))
            frames.append(img)
        return np.stack(frames)

    else:  # video_file — decord-ით პირდაპირ target frame-ებზე ვახტებით
        try:
            vr = VideoReader(row['path'], ctx=cpu(0))
        except Exception:
            return None
        total = len(vr)
        if total == 0:
            return None
        idxs = np.linspace(0, total - 1, num_frames).astype(int).tolist()
        batch = vr.get_batch(idxs).asnumpy()  # (T, H, W, C) RGB უკვე
        frames = [cv2.resize(f, (size, size)) for f in batch]
        return np.stack(frames)

In [24]:
# CELL 12 — cache-ის გენერაცია
from tqdm import tqdm

failed = []
for idx, row in tqdm(video_meta.iterrows(), total=len(video_meta)):
    out_path = os.path.join(CACHE_DIR, f"{row['video_id']}.npy")
    if os.path.exists(out_path):
        continue
    try:
        frames = load_uniform_frames(row)
        if frames is None:
            failed.append(row['video_id'])
            continue
        np.save(out_path, frames)
    except Exception:
        failed.append(row['video_id'])

print(f"ჩავარდა: {len(failed)} / {len(video_meta)}")

100%|██████████| 1806/1806 [23:35<00:00,  1.28it/s]

ჩავარდა: 0 / 1806


In [25]:
# CELL 13 — manifest-ის დასრულება + Drive-ზე გაზიარება
manifest_clean = video_meta[~video_meta['video_id'].isin(failed)].copy()
manifest_clean['cache_path'] = manifest_clean['video_id'].apply(
    lambda vid: os.path.join(CACHE_DIR, f'{vid}.npy'))

manifest_clean.to_csv('/content/drive/MyDrive/Ketastasia/data/combined_cached_final.csv', index=False)

!tar -czf /content/frame_cache.tar.gz -C /content frame_cache
!cp /content/frame_cache.tar.gz '/content/drive/MyDrive/Ketastasia/data/'
print("Cache მზადაა და Drive-ზეა — მეორე ადამიანმა ჩამოშალოს იქიდან.")

Cache მზადაა და Drive-ზეა — მეორე ადამიანმა ჩამოშალოს იქიდან.
